### Models

#### Invoking Model

In [ ]:
from langchain.chat_models import init_chat_model
# from dotenv import load_dotenv
# load_dotenv()
import os, sys

# os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model(
    model="ollama:qwen2.5:7b",
    temperature = 0.5,
    max_tokens = 2500,
    max_retries = 2,
    timeout= 30,
)

response = model.invoke("What is a parrot ?")
print(response.content)

#### Streaming

In [39]:
stream = model.stream("Which is the most powerful animal in entire world")
for chunk in stream:
    print(chunk.text, end="|", flush=True)

Determin|ing| the| "|most| powerful|"| animal| in| the| world| can| be| subjective| and| depends| on| the| criteria| used|.| Here| are| a| few| perspectives|:

|1|.| **|Physical| Strength|**:| By| this| measure|,| some| of| the| strongest| animals| include| elephants|,| gor|illas|,| and| hip|pos|.| However|,| when| considering| sheer| power|-to|-body|-m|ass| ratios|,| animals| like| the| African| lion| or| even| insects| like| the| Hercules| beetle| can| be| considered| powerful|.

|2|.| **|Pred|atory| Ability|**:| In| terms| of| hunting| prowess|,| apex| predators| like| the| great| white| shark|,| lions|,| t|igers|,| and| cro|cod|iles| are| often| cited|.| These| animals| are| highly| efficient| hunters| and| can| take| down| large| prey|.

|3|.| **|Impact| on| E|cosystem|**:| Ele|phants|,| bees|,| and| ants| are| considered| powerful| due| to| their| significant| impact| on| their| ecosystems|.| Be|es|,| for| example|,| are| crucial| for| poll|ination|,| and| ants| can| alter| lands

#### Batching

##### With Entire Output

In [40]:
batch_response = model.batch([
    "What is purpose of 2 kidneys in human body?",
    "Why elephants can not fly?",
    "Why mens have beard but women have not ?"
])

for response in batch_response:
    print(response.text)

The primary purpose of having two kidneys in the human body is to maintain a balanced internal environment, a process known as homeostasis. Each kidney performs the same basic functions, but having two provides redundancy and ensures that the body can function normally even if one kidney is not working properly. Here are the key functions of the kidneys:

1. **Filtering Blood**: The kidneys filter about 200 liters of blood every day to remove waste products, excess water, and other impurities. This process helps to maintain the purity of the blood.

2. **Regulating Fluid Balance**: By controlling the amount of water and electrolytes (such as sodium, potassium, and calcium) in the body, the kidneys help to maintain the right balance of these substances. This is crucial for the proper functioning of cells and organs.

3. **Producing Hormones**: The kidneys produce several important hormones, including erythropoietin (which stimulates red blood cell production), renin (which helps regulat

##### When Completed

In [41]:
batch_response = model.batch_as_completed([
    "What is purpose of 2 kidneys in human body?",
    "Why elephants can not fly?",
    "Why mens have beard but women have not ?"
])

print(batch_response)

<generator object Runnable.batch_as_completed at 0x000001FD9C3B7940>


### Tools (Invoking Tools)

In [45]:
from langchain.tools import tool
import getpass

@tool
def getUserName() -> str:
    "Get windows user name"
    return f"User Name {getpass.getuser()}"
tools = [getUserName]
tools_by_name = {tool.name: tool for tool in tools}
model_with_tools = model.bind_tools(tools)
messages = [{"role":"user", "content":"what this current user name"}]
ai_response = model_with_tools.invoke(messages)
messages.append(ai_response)
for tool_call in ai_response.tool_calls:
    toolInfo = tools_by_name[tool_call["name"]]
    tool_result = toolInfo.invoke(tool_call)
    messages.append(tool_result)
final_response = model_with_tools.invoke(messages)
print(final_response.text)


The current user name is mahen.


#### Structured output

In [ ]:
from pydantic import BaseModel, Field

class BotMessage(BaseModel):
    calorie: str = Field(description="Total Amount Of calorie to be taken")
    weight: int = Field(description="Weight of the person in lbs/kg")
    diet_type: str = Field(description="Keto Diet or Normal Diet")
    protein_intake: str = Field(description="Amount of protein according to body weight")

model_with_structure = model.with_structured_output(schema=BotMessage)
model_with_structure.invoke(f"Please preapare a plan for making me fit with in 90 days, if i am a 5.6ft and having 65kg body weight")

BotMessage(calorie='1500-1800', weight=65, diet_type='balanced', protein_intake='1.2 g/kg body weight (78g per day for 65kg person)')

In [49]:
user_prompt = "Please preapare a plan for making me fit with in 90 days, if i am a 5.6ft and having 65kg body weight"
ms = model.with_structured_output(schema=BotMessage, include_raw=True)
response = ms.invoke(user_prompt)
response

{'raw': AIMessage(content='{\n  "calorie" : "deficit",\n  "weight" : 65,\n  "diet_type" : "balanced",\n  "protein_intake" : "adequate_for_fitness_goals"\n  }', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-10-03T16:30:04.783634797Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7952673274, 'load_duration': 3217233, 'prompt_eval_count': 63, 'prompt_eval_duration': 307170000, 'eval_count': 46, 'eval_duration': 7611456000, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--01a10299-ced6-7bf2-8384-91e865637837-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 63, 'output_tokens': 46, 'total_tokens': 109}),
 'parsed': BotMessage(calorie='deficit', weight=65, diet_type='balanced', protein_intake='adequate_for_fitness_goals'),
 'parsing_error': None}